# Automatic Download of ClusPro finished jobs

## Load Job list

### Option 1: Use a csv list of the jobs

In [3]:
import pandas as pd
df = pd.read_csv('./job_list.csv', sep='\t') # update with your csv job file
df.head()



,Protein,JobID
0,Example1,1410658
1,Example2,1410659


In [ ]:
job_ids = df['JobID'] # Change for the correct name column

### Option 2: Use a list of jobs directly

In [ ]:
# -------- CONFIG --------
#job_ids = [1410658, 1410659] # Update with your job list if not defined before

## Download files

### Get your cookie from ClusPro in Web Browser


In [5]:
PHPSESSID = "f2fbff8c2ee67a6adff3da1bf1d7154b"  # <-- your cookie session


### Run Download and decompress .tar.bz2 files

In [6]:
import os
import time
import requests
import tarfile
import urllib3

coefficients = [0, 2, 4, 6]

output_dir = "cluspro_downloads"
os.makedirs(output_dir, exist_ok=True)

MAX_RETRIES = 5
SLEEP_BETWEEN_RETRIES = 5
REQUEST_DELAY = 0.5
DELETE_BZ2 = False # Change to True if you don need the compressed files

# Disable SSL warnings (server has broken cert)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


# -------- SESSION --------
session = requests.Session()
session.verify = False

session.cookies.update({
    "PHPSESSID": PHPSESSID
})

headers = {
    "User-Agent": "Mozilla/5.0"
}


# -------- HELPERS --------
def is_valid_file(path, min_size=100):
    return os.path.exists(path) and os.path.getsize(path) > min_size


def is_bz2_file(path):
    try:
        with open(path, "rb") as f:
            return f.read(3) == b'BZh'
    except:
        return False


def download_with_retries(url, save_path):
    if is_valid_file(save_path):
        print(f"Skipping: {save_path}")
        return True

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(url, headers=headers, stream=True, timeout=30)

            if response.status_code != 200:
                raise Exception(f"HTTP {response.status_code}")

            content_type = response.headers.get("Content-Type", "")
            if "text/html" in content_type:
                raise Exception("HTML response (job not ready or auth failed)")

            tmp_path = save_path + ".tmp"

            with open(tmp_path, "wb") as f:
                for chunk in response.iter_content(8192):
                    f.write(chunk)

            if not is_valid_file(tmp_path):
                raise Exception("File too small / corrupted")

            os.rename(tmp_path, save_path)
            print(f"Downloaded: {save_path}")
            return True

        except Exception as e:
            print(f"[Attempt {attempt}] Failed: {url}")
            print(f"Reason: {e}")

            if attempt < MAX_RETRIES:
                time.sleep(SLEEP_BETWEEN_RETRIES * attempt)
            else:
                print(f"Gave up: {url}")
                return False

        finally:
            if os.path.exists(save_path + ".tmp"):
                os.remove(save_path + ".tmp")


def decompress_bz2(file_path):
    output_path = file_path.replace(".tar.bz2", "")
    if is_valid_file(output_path):
        print(f"Skipping decompression: {output_path}")
        return

    if not is_bz2_file(file_path):
        print(f"Invalid bz2 file: {file_path}")
        os.remove(file_path)
        return

    try:
        with tarfile.open(file_path, "r:bz2") as tar:
            tar.extractall(path=output_path)
        print(f"Decompressed: {output_path}")

        if DELETE_BZ2:
            os.remove(file_path)

    except Exception as e:
        print(f"Decompression failed: {file_path}")
        print(e)
        os.remove(file_path)


# -------- MAIN LOOP --------
for job in job_ids:
    print(f"\n===== JOB {job} =====")
    job_dir = os.path.join(output_dir, f"job_{job}")
    os.makedirs(job_dir, exist_ok=True)

    # --- MODEL ---
    model_url = f"https://cluspro.org/file.php?jobid={job}&coeffi=0&model=0&filetype=model_bz2"
    model_path = os.path.join(job_dir, f"cluspro.{job}.tar.bz2")
    success = download_with_retries(model_url, model_path)

    if success:
        decompress_bz2(model_path)

    time.sleep(REQUEST_DELAY)

    # --- SCORES ---
    for coeff in coefficients:
        score_url = f"https://cluspro.org/scores_csv.php?job={job}&coeffi={coeff}"
        score_path = os.path.join(job_dir, f"cluspro_scores.{job}.00{coeff}.csv")

        download_with_retries(score_url, score_path)
        time.sleep(REQUEST_DELAY)


===== JOB 1410658 =====
Downloaded: cluspro_downloads/job_1410658/cluspro.1410658.tar.bz2
Decompressed: cluspro_downloads/job_1410658/cluspro.1410658
Downloaded: cluspro_downloads/job_1410658/cluspro_scores.1410658.000.csv
Downloaded: cluspro_downloads/job_1410658/cluspro_scores.1410658.002.csv
Downloaded: cluspro_downloads/job_1410658/cluspro_scores.1410658.004.csv
Downloaded: cluspro_downloads/job_1410658/cluspro_scores.1410658.006.csv

===== JOB 1410659 =====
Downloaded: cluspro_downloads/job_1410659/cluspro.1410659.tar.bz2
Decompressed: cluspro_downloads/job_1410659/cluspro.1410659
Downloaded: cluspro_downloads/job_1410659/cluspro_scores.1410659.000.csv
Downloaded: cluspro_downloads/job_1410659/cluspro_scores.1410659.002.csv
Downloaded: cluspro_downloads/job_1410659/cluspro_scores.1410659.004.csv
Downloaded: cluspro_downloads/job_1410659/cluspro_scores.1410659.006.csv
